# CAWOT-CM V0 — Kaggle end-to-end (HuggingFace direct)

**Source of truth: HuggingFace** [`TruongVox/Cawot-dataset`](https://huggingface.co/datasets/TruongVox/Cawot-dataset).
Notebook downloads N shards (image zips + JSONL annotations) from HF and runs V0.

**V0 plan (clean baseline, no qproxy):**
1. Download N shards from HF, extract
2. Random sample 50K (image, caption) entries from the loaded pool
3. Extract CLIP ViT-B/16 embeddings
4. Hold out 2K pairs as val
5. FAISS k-means K=150 over 48K train pool
6. Two coresets at 20% budget: Random vs V0 (farthest-from-centroid)
7. Fine-tune CLIP-B/16 on each
8. Image-text retrieval R@1/5/10 on val

**Setup (Kaggle UI):**
- Settings → Accelerator: **GPU P100**
- Settings → Internet: **On**
- No external dataset needed — notebook downloads from HF.

Disk budget: 5 shards ≈ 7 GB download + ~14 GB extracted ≈ 21 GB peak. Kaggle has ~20 GB persistent + 50 GB ephemeral, so this fits.

## 1. Sanity check env

In [ ]:
!nvidia-smi -L
!df -h /kaggle/working | tail -1

## 2. Clone repo + install deps

In [ ]:
import os
if not os.path.exists("/kaggle/working/cawot-cm"):
    !git clone https://github.com/HohoHocCode/cawot-cm.git /kaggle/working/cawot-cm
%cd /kaggle/working/cawot-cm
!pip install -q open_clip_torch faiss-gpu-cu12 einops huggingface_hub

## 3. Download N shards from HuggingFace

Each shard ≈ 1.4 GB zip + ~5 MB JSON. 5 shards ≈ 7 GB download, ~14 GB extracted.

Bump `NUM_SHARDS` if you want a larger pool (longer download + more disk).

In [ ]:
NUM_SHARDS = 5
!python scripts/setup_data.py --output /kaggle/working/pab_data --num-shards {NUM_SHARDS}

## 4. Sanity check that one image resolves

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/cawot-cm")
from src.data import build_pool, TrainPoolDataset
from torchvision import transforms

anns, shard_roots = build_pool(
    annotations_dir="/kaggle/working/pab_data/annotations",
    image_root="/kaggle/working/pab_data/images",
    sample_size=100,
    seed=42,
)
print(f"loaded {len(anns)} annotations")
print(f"shards on disk: {sorted(shard_roots)}")

tx = transforms.Compose([transforms.Resize(224), transforms.CenterCrop(224), transforms.ToTensor()])
ds = TrainPoolDataset(anns, shard_roots, image_transform=tx)
sample = ds[0]
print(f"\nfirst image: shape={sample['image'].shape}")
print(f"first caption: {sample['caption'][:80]}...")
print("\n✓ pipeline resolves images correctly")

## 5. Run V0 end-to-end

On Kaggle P100 with 50K pool, 20% budget:
- Extract CLIP embeddings (50K): ~15-20 min
- Cluster (48K, K=150): ~30 sec
- Train each coreset (~9.6K × 3 epochs): ~15 min each
- Eval each: ~2 min
- **Total: ~50-70 min**

All intermediate outputs cached under `/kaggle/working/outputs/` — safe to interrupt and re-run.

In [ ]:
!python scripts/run_v0.py --config config.yaml

## 6. Results

In [ ]:
import json, pandas as pd
with open("/kaggle/working/outputs/eval/summary.json") as f:
    summary = json.load(f)
df = pd.DataFrame(summary).T
df

## 7. What to look for

- **`mean_R@1`**: avg of t2i and i2t recall at 1 on 2K held-out val pairs.
- **Expected ordering**: `v0 > random > zeroshot`. Typical V0-over-Random gap: 0.5–1.5%.
- **If V0 ≤ Random**: bug in selection or K is wrong scale. Sanity check before V1.
- **If both barely beat zeroshot**: training collapsed. Inspect LR, batch size.

Click **Save Version** → **Save & Run All** to persist `outputs/` as the notebook output so checkpoints + summary can be downloaded for review.